# 취약권역 K-means 보조 분석

- 목적: DBSCAN으로 생성한 종합취약권역을 대상으로 K-means 유형화를 시도하고 군집 구조가 뚜렷한지 확인함.
- 범위: `K=3~8`의 엘보우와 실루엣 점수를 확인하되, 최종 대시보드 유형 분류로 채택하지 않음.
- 입력: 현재 `vulnerability_index` 폴더의 DBSCAN 상위 10% 산출물을 읽음.
- 저장: 기본 실행에서는 K-means CSV와 이미지를 새로 저장하지 않음.
- 기존 저장 산출물은 04 기반 보조 분석/실패 근거로 보존함.

In [ ]:
from pathlib import Path
import os
import sys
import warnings

os.environ["LOKY_MAX_CPU_COUNT"] = "1"

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8")

warnings.filterwarnings(
    "ignore",
    message="Could not find the number of physical cores.*",
    category=UserWarning,
)

import matplotlib
matplotlib.use("Agg")
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


def display(data):
    if isinstance(data, pd.DataFrame):
        print(data.to_string(index=False))
    elif isinstance(data, pd.Series):
        print(data.to_string())
    else:
        print(data)


BASE_PATH = Path.cwd().resolve()

if BASE_PATH.name == "dashboard":
    PROJECT_PATH = BASE_PATH.parents[1]
elif BASE_PATH.name == "notebooks":
    PROJECT_PATH = BASE_PATH.parent
elif (BASE_PATH / "notebooks").exists():
    PROJECT_PATH = BASE_PATH
else:
    PROJECT_PATH = BASE_PATH

DASHBOARD_PATH = PROJECT_PATH / "notebooks" / "dashboard"
VULNERABILITY_PATH = DASHBOARD_PATH / "OUTPUT" / "vulnerability_index"
ACCESS_PATH = PROJECT_PATH / "notebooks" / "access" / "OUTPUT" / "h3sfca"
DEMOGRAPHIC_PATH = (
    PROJECT_PATH
    / "analysis_table"
    / "data"
    / "output"
    / "서울시_격자_100m_문화누리대상자_성연령장애별_인구수.csv"
)
OUTPUT_PATH = DASHBOARD_PATH / "OUTPUT" / "vulnerability_kmeans"
IMAGE_PATH = DASHBOARD_PATH / "IMAGE" / "vulnerability_kmeans"
SAVE_FIGURE = False
SAVE_TABLE = False
SAVE_FINAL_TABLES = False
SAVE_MAP_IMAGES = False
PROFILE_K_LIST = [3, 4]
FINAL_K = 4
USE_DEMOGRAPHIC_PROFILE = False

if SAVE_FIGURE or SAVE_TABLE or SAVE_FINAL_TABLES:
    OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
if SAVE_FIGURE or SAVE_MAP_IMAGES:
    IMAGE_PATH.mkdir(parents=True, exist_ok=True)

FONT_PATH = Path("C:/Windows/Fonts/NotoSansKR-VF.ttf")
if FONT_PATH.exists():
    fm.fontManager.addfont(str(FONT_PATH))
    plt.rcParams["font.family"] = "Noto Sans KR"
else:
    plt.rcParams["font.family"] = "Malgun Gothic"

plt.rcParams["axes.unicode_minus"] = False

THEME_ORANGE = "#ff641d"
THEME_LIGHT_GRAY = "#dedbd5"
THEME_DARK = "#2f2a26"

print("PROJECT_PATH:", PROJECT_PATH)
print("VULNERABILITY_PATH:", VULNERABILITY_PATH)
print("ACCESS_PATH:", ACCESS_PATH)
print("OUTPUT_PATH:", OUTPUT_PATH)
print("IMAGE_PATH:", IMAGE_PATH)
print("SAVE_FIGURE:", SAVE_FIGURE)
print("SAVE_TABLE:", SAVE_TABLE)
print("SAVE_FINAL_TABLES:", SAVE_FINAL_TABLES)
print("SAVE_MAP_IMAGES:", SAVE_MAP_IMAGES)
print("FINAL_K:", FINAL_K)
print("USE_DEMOGRAPHIC_PROFILE:", USE_DEMOGRAPHIC_PROFILE)



## 1. 입력자료 불러오기

- 입력: 현재 `OUTPUT/vulnerability_index`의 `DBSCAN_취약권역.csv`, `DBSCAN_취약격자.csv`
- 분류별 시설 접근성 입력: `h3sfca_격자_중분류_접근성.csv`
- 분석대상: `권역유형 == "종합취약"`
- 제외: 하위지표별 권역은 K-means 후보 비교에서 제외함.
- 주의: 이 노트북은 보조 분석용이며, 기본 설정에서는 기존 K-means 산출물을 덮어쓰지 않음.

In [ ]:
cluster_path = VULNERABILITY_PATH / "DBSCAN_취약권역.csv"
grid_path = VULNERABILITY_PATH / "DBSCAN_취약격자.csv"
access_path = ACCESS_PATH / "h3sfca_격자_중분류_접근성.csv"

if not cluster_path.exists():
    raise FileNotFoundError(f"DBSCAN 취약권역 파일이 없습니다: {cluster_path}")
if not grid_path.exists():
    raise FileNotFoundError(f"DBSCAN 취약격자 파일이 없습니다: {grid_path}")
if not access_path.exists():
    raise FileNotFoundError(f"중분류별 H3SFCA 접근성 파일이 없습니다: {access_path}")

cluster = pd.read_csv(cluster_path)
grid = pd.read_csv(
    grid_path,
    usecols=[
        "권역유형",
        "GRID_CD",
        "취약권역_ID",
        "DBSCAN_label",
        "기준취약점수",
        "중심점_x",
        "중심점_y"
    ]
)
target = cluster[cluster["권역유형"].eq("종합취약")].copy()

print("전체 취약권역:", cluster.shape)
print("종합취약권역:", target.shape)
print("전체 취약격자:", grid.shape)

display(target.head())



## 2. K-means 입력변수 설계

- 최종취약지수는 군집 입력에서 제외함.
- 통합 시설 접근성 대신 중분류별 시설 접근성 취약도를 사용함.
- 문화다양성, 장애인친화, 노인편의 취약도와 권역 규모, 대상자 밀도, 장애인·노령인구 비율을 함께 사용함.
- 성비와 연령대 성비는 군집 구분력이 낮아 최종 프로파일링에서도 제외함.
- 04_1 반영 후 재실행하더라도 기본 설정에서는 결과 파일을 저장하지 않음.

In [ ]:
def zscore_by_mask(series, mask):
    result = pd.Series(np.nan, index=series.index, dtype="float64")
    values = pd.to_numeric(series, errors="coerce")
    mean_value = values.loc[mask].mean()
    std_value = values.loc[mask].std(ddof=0)

    if pd.isna(std_value) or np.isclose(std_value, 0):
        result.loc[mask] = 0
    else:
        result.loc[mask] = (values.loc[mask] - mean_value) / std_value

    return result


access = pd.read_csv(
    access_path,
    encoding="utf-8-sig",
    usecols=["GRID_CD", "중분류", "접근성지수", "문화누리대상자_추정_인구수"]
)

access["접근성지수"] = pd.to_numeric(access["접근성지수"], errors="coerce").fillna(0)
access["문화누리대상자_추정_인구수"] = pd.to_numeric(
    access["문화누리대상자_추정_인구수"],
    errors="coerce"
).fillna(0)
access["분석대상"] = access["문화누리대상자_추정_인구수"].gt(0)
access["중분류별_시설접근성취약_z"] = np.nan

for category in sorted(access["중분류"].dropna().unique()):
    idx = access["중분류"].eq(category)
    access.loc[idx, "중분류별_시설접근성취약_z"] = -zscore_by_mask(
        access.loc[idx, "접근성지수"],
        access.loc[idx, "분석대상"]
    )

target_grid = grid[
    grid["권역유형"].eq("종합취약")
    & grid["취약권역_ID"].isin(target["취약권역_ID"])
].copy()

category_access = target_grid[["취약권역_ID", "GRID_CD"]].merge(
    access[["GRID_CD", "중분류", "중분류별_시설접근성취약_z"]],
    on="GRID_CD",
    how="left"
)

category_profile = (
    category_access
    .groupby(["취약권역_ID", "중분류"], as_index=False)
    .agg(평균_분류별_접근성취약도=("중분류별_시설접근성취약_z", "mean"))
)

category_profile = (
    category_profile
    .pivot(index="취약권역_ID", columns="중분류", values="평균_분류별_접근성취약도")
    .add_prefix("평균_")
    .add_suffix("_접근성취약도")
    .reset_index()
)

facility_category_features = [
    col for col in category_profile.columns
    if col != "취약권역_ID"
]

target = target.merge(category_profile, on="취약권역_ID", how="left")

print("중분류별 시설 접근성 입력 변수")
print(pd.Series(facility_category_features).to_string(index=False))
print()
print("종합취약 권역 포함 격자 수:", target_grid["GRID_CD"].nunique())

target["문화누리대상자_밀도_ha"] = (
    target["문화누리대상자_추정인구수"] / target["권역면적_m2"].replace(0, np.nan) * 10000
)
target["장애인비율"] = (
    target["장애인_수요인구수"] / target["문화누리대상자_추정인구수"].replace(0, np.nan)
)
target["노령인구비율"] = (
    target["노령인구_수요인구수"] / target["문화누리대상자_추정인구수"].replace(0, np.nan)
)
target["권역규모_log"] = np.log1p(target["포함_취약격자수"])

target["장애인비율"] = target["장애인비율"].clip(lower=0, upper=1)
target["노령인구비율"] = target["노령인구비율"].clip(lower=0, upper=1)

density_q99 = target["문화누리대상자_밀도_ha"].quantile(0.99)
target["문화누리대상자_밀도_ha"] = target["문화누리대상자_밀도_ha"].clip(upper=density_q99)

kmeans_features = facility_category_features + [
    "평균_문화다양성부족도",
    "평균_장애인친화시설_접근성취약도",
    "평균_노인편의서비스_접근성취약도",
    "권역규모_log",
    "문화누리대상자_밀도_ha",
    "장애인비율",
    "노령인구비율",
]

missing_features = [col for col in kmeans_features if col not in target.columns]
if missing_features:
    raise KeyError(f"K-means 입력변수가 없습니다: {missing_features}")

kmeans_input = target[["취약권역_ID"] + kmeans_features].copy()
na_summary = kmeans_input[kmeans_features].isna().sum()

print("입력변수 결측 수")
print(na_summary.to_string())

kmeans_input = kmeans_input.dropna(subset=kmeans_features).copy()

print()
print("K-means 입력 권역 수:", len(kmeans_input))
display(kmeans_input[kmeans_features].describe().T)



## 3. 표준화 및 품질 점검

- K-means는 변수 단위에 민감하므로 모든 입력변수를 표준화함.
- 권역 규모와 대상자 밀도는 분포가 치우쳐 있어 로그 변환과 99퍼센타일 상한을 적용함.



In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(kmeans_input[kmeans_features])

scaled_check = pd.DataFrame(X, columns=kmeans_features)

print("표준화 후 평균")
print(scaled_check.mean().round(6).to_string())

print()
print("표준화 후 표준편차")
print(scaled_check.std(ddof=0).round(6).to_string())



## 4. K 후보 비교

- 후보 K: `3~8`
- 확인 지표: 엘보우용 inertia, 실루엣 점수, 군집 크기 균형
- 최종 K 확정은 하지 않음.



In [ ]:
k_candidates = range(3, 9)
k_rows = []

for k in k_candidates:
    model = KMeans(n_clusters=k, random_state=42, n_init=100)
    labels = model.fit_predict(X)
    sizes = pd.Series(labels).value_counts().sort_index()

    k_rows.append({
        "K": k,
        "inertia": model.inertia_,
        "silhouette_score": silhouette_score(X, labels),
        "최소군집권역수": int(sizes.min()),
        "최대군집권역수": int(sizes.max()),
        "평균군집권역수": float(sizes.mean()),
        "군집크기_CV": float(sizes.std(ddof=0) / sizes.mean())
    })

k_compare = pd.DataFrame(k_rows)
k_compare["inertia_감소율"] = k_compare["inertia"].pct_change().mul(-100)

print("K 후보 비교")
display(k_compare)



## 5. 엘보우 및 실루엣 그래프

- 주황색은 주요 비교선, 옅은 회색은 보조 비교선으로 사용함.
- 그래프에는 불필요한 설명문을 넣지 않음.
- `SAVE_FIGURE`가 `True`이면 이미지 파일만 저장함.



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), dpi=160)

axes[0].plot(
    k_compare["K"],
    k_compare["inertia"],
    marker="o",
    linewidth=2.4,
    color=THEME_ORANGE
)
axes[0].bar(
    k_compare["K"],
    k_compare["inertia"],
    color=THEME_LIGHT_GRAY,
    alpha=0.35,
    width=0.55,
    zorder=0
)
axes[0].set_title("Elbow", fontsize=13, color=THEME_DARK, pad=10)
axes[0].set_xlabel("K")
axes[0].set_ylabel("Inertia")

axes[1].plot(
    k_compare["K"],
    k_compare["silhouette_score"],
    marker="o",
    linewidth=2.4,
    color=THEME_ORANGE
)
axes[1].bar(
    k_compare["K"],
    k_compare["silhouette_score"],
    color=THEME_LIGHT_GRAY,
    alpha=0.35,
    width=0.55,
    zorder=0
)
axes[1].set_title("Silhouette", fontsize=13, color=THEME_DARK, pad=10)
axes[1].set_xlabel("K")
axes[1].set_ylabel("Score")

for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#b9aa9d")
    ax.spines["bottom"].set_color("#b9aa9d")
    ax.tick_params(colors=THEME_DARK)
    ax.grid(axis="y", color="#eee7df", linewidth=0.8)

plt.tight_layout()

if SAVE_FIGURE:
    figure_path = IMAGE_PATH / "kmeans_category_elbow_silhouette.png"
    fig.savefig(figure_path, bbox_inches="tight", facecolor="white")
    print("저장:", figure_path)
else:
    print("이미지 저장 안 함: SAVE_FIGURE=False")

plt.close(fig)



## 6. 산출물 저장 여부

- 기본값은 저장하지 않음.
- K별 권역 라벨은 최종 K를 정한 뒤 별도 생성함.



In [ ]:
if SAVE_TABLE:
    compare_path = OUTPUT_PATH / "Kmeans_K후보비교.csv"
    k_compare.to_csv(compare_path, index=False, encoding="utf-8-sig")
    print("저장:", compare_path)
else:
    print("표 저장 안 함: SAVE_TABLE=False")



## 7. 결과 요약

- 실루엣 점수와 엘보우 변화량을 함께 보고 후보 K를 판단함.
- 군집 크기가 지나치게 불균형한 K는 해석 가능성을 추가 확인해야 함.
- 성비와 연령대 성비는 다음 단계의 군집 프로파일링에서만 붙임.



In [ ]:
best_silhouette = k_compare.loc[k_compare["silhouette_score"].idxmax()].copy()

print("실루엣 최고 K:", int(best_silhouette["K"]))
print("실루엣 최고 점수:", round(best_silhouette["silhouette_score"], 4))

print()
print("K별 요약")
display(k_compare[[
    "K",
    "inertia",
    "inertia_감소율",
    "silhouette_score",
    "최소군집권역수",
    "최대군집권역수",
    "군집크기_CV"
]])



## 8. K=3·K=4 군집 라벨 생성

- 프로파일링 비교 대상은 `K=3`, `K=4`로 고정함.
- K-means 원래 라벨은 임의 번호이므로 권역 수가 많은 순서로 `유형1`, `유형2` 형태로 재부여함.
- 라벨은 분석 중 확인용이며 별도 파일로 저장하지 않음.



In [ ]:
profile = target[target["취약권역_ID"].isin(kmeans_input["취약권역_ID"])].copy()


def relabel_by_cluster_size(labels, label_prefix):
    size_order = (
        pd.Series(labels)
        .value_counts()
        .sort_values(ascending=False)
        .index
        .tolist()
    )
    label_map = {
        old_label: f"{label_prefix}_유형{order + 1}"
        for order, old_label in enumerate(size_order)
    }
    return pd.Series(labels).map(label_map).to_numpy()


for profile_k in PROFILE_K_LIST:
    model = KMeans(n_clusters=profile_k, random_state=42, n_init=100)
    raw_labels = model.fit_predict(X)
    label_col = f"K{profile_k}_유형"

    label_frame = pd.DataFrame({
        "취약권역_ID": kmeans_input["취약권역_ID"].to_numpy(),
        label_col: relabel_by_cluster_size(raw_labels, f"K{profile_k}")
    })

    profile = profile.merge(label_frame, on="취약권역_ID", how="left")

print("K=3 유형 분포")
print(profile["K3_유형"].value_counts().sort_index().to_string())

print()
print("K=4 유형 분포")
print(profile["K4_유형"].value_counts().sort_index().to_string())



## 9. 성별·연령대별 프로파일링 제외

- 성별·연령대 변수는 군집 구분력이 약해 이번 프로파일링 출력에서 제외함.
- 필요한 경우 `USE_DEMOGRAPHIC_PROFILE=True`로 바꿔 별도 검토할 수 있음.
- 기본 실행에서는 성연령 원자료를 읽지 않아 처리 시간을 줄임.



In [ ]:
AGE_GROUP_MAP = {
    "0-5세": "0_19세",
    "6-14세": "0_19세",
    "15-19세": "0_19세",
    "20-29세": "20_39세",
    "30-39세": "20_39세",
    "40-49세": "40_59세",
    "50-59세": "40_59세",
    "60-69세": "60세이상",
    "70-79세": "60세이상",
    "80-89세": "60세이상",
    "90-99세": "60세이상",
    "100세-": "60세이상"
}

AGE_GROUPS = ["0_19세", "20_39세", "40_59세", "60세이상"]


def build_demographic_profile(demographic_path, region_grid):
    if not demographic_path.exists():
        print("성연령 원자료 없음:", demographic_path)
        return pd.DataFrame({"취약권역_ID": region_grid["취약권역_ID"].drop_duplicates()})

    usecols = [
        "GRID_CD",
        "성별",
        "연령대",
        "문화누리대상자_성연령별_추정_인구수"
    ]
    target_grid_codes = set(region_grid["GRID_CD"].astype(str))
    filtered_chunks = []

    for chunk in pd.read_csv(
        demographic_path,
        encoding="utf-8-sig",
        usecols=usecols,
        chunksize=300000
    ):
        chunk["GRID_CD"] = chunk["GRID_CD"].astype(str)
        chunk = chunk[chunk["GRID_CD"].isin(target_grid_codes)].copy()
        if not chunk.empty:
            filtered_chunks.append(chunk)

    if not filtered_chunks:
        print("성연령 원자료와 매칭되는 권역 격자가 없음")
        return pd.DataFrame({"취약권역_ID": region_grid["취약권역_ID"].drop_duplicates()})

    demographic = pd.concat(filtered_chunks, ignore_index=True)
    value_col = "문화누리대상자_성연령별_추정_인구수"
    demographic[value_col] = pd.to_numeric(demographic[value_col], errors="coerce").fillna(0)
    demographic["연령그룹"] = demographic["연령대"].map(AGE_GROUP_MAP)
    demographic = demographic[demographic["연령그룹"].notna()].copy()

    region_map = region_grid[["취약권역_ID", "GRID_CD"]].drop_duplicates().copy()
    region_map["GRID_CD"] = region_map["GRID_CD"].astype(str)

    demographic = demographic.merge(region_map, on="GRID_CD", how="inner")

    total = (
        demographic
        .groupby("취약권역_ID", as_index=False)[value_col]
        .sum()
        .rename(columns={value_col: "성연령_문화누리대상자수"})
    )

    sex_wide = (
        demographic
        .groupby(["취약권역_ID", "성별"], as_index=False)[value_col]
        .sum()
        .pivot(index="취약권역_ID", columns="성별", values=value_col)
        .fillna(0)
        .reset_index()
        .rename(columns={
            "남성": "남성_문화누리대상자수",
            "여성": "여성_문화누리대상자수"
        })
    )

    for col in ["남성_문화누리대상자수", "여성_문화누리대상자수"]:
        if col not in sex_wide.columns:
            sex_wide[col] = 0

    age_wide = (
        demographic
        .groupby(["취약권역_ID", "연령그룹"], as_index=False)[value_col]
        .sum()
        .pivot(index="취약권역_ID", columns="연령그룹", values=value_col)
        .fillna(0)
        .reset_index()
        .rename(columns={age: f"{age}_문화누리대상자수" for age in AGE_GROUPS})
    )

    for age in AGE_GROUPS:
        col = f"{age}_문화누리대상자수"
        if col not in age_wide.columns:
            age_wide[col] = 0

    age_sex = (
        demographic
        .groupby(["취약권역_ID", "연령그룹", "성별"], as_index=False)[value_col]
        .sum()
    )

    female_age = age_sex[age_sex["성별"].eq("여성")].copy()
    female_age = female_age.rename(columns={value_col: "연령그룹_여성수"})

    age_total = (
        age_sex
        .groupby(["취약권역_ID", "연령그룹"], as_index=False)[value_col]
        .sum()
        .rename(columns={value_col: "연령그룹_합계"})
    )

    female_age_ratio = age_total.merge(
        female_age[["취약권역_ID", "연령그룹", "연령그룹_여성수"]],
        on=["취약권역_ID", "연령그룹"],
        how="left"
    )
    female_age_ratio["연령그룹_여성수"] = female_age_ratio["연령그룹_여성수"].fillna(0)
    female_age_ratio["연령그룹_여성비율"] = np.where(
        female_age_ratio["연령그룹_합계"].gt(0),
        female_age_ratio["연령그룹_여성수"] / female_age_ratio["연령그룹_합계"],
        np.nan
    )

    female_age_ratio = (
        female_age_ratio
        .pivot(index="취약권역_ID", columns="연령그룹", values="연령그룹_여성비율")
        .reset_index()
        .rename(columns={age: f"{age}_여성비율" for age in AGE_GROUPS})
    )

    demographic_profile = (
        total
        .merge(sex_wide, on="취약권역_ID", how="left")
        .merge(age_wide, on="취약권역_ID", how="left")
        .merge(female_age_ratio, on="취약권역_ID", how="left")
    )

    demographic_profile["남성비율"] = np.where(
        demographic_profile["성연령_문화누리대상자수"].gt(0),
        demographic_profile["남성_문화누리대상자수"] / demographic_profile["성연령_문화누리대상자수"],
        np.nan
    )
    demographic_profile["여성비율"] = np.where(
        demographic_profile["성연령_문화누리대상자수"].gt(0),
        demographic_profile["여성_문화누리대상자수"] / demographic_profile["성연령_문화누리대상자수"],
        np.nan
    )

    for age in AGE_GROUPS:
        count_col = f"{age}_문화누리대상자수"
        ratio_col = f"{age}_비율"
        demographic_profile[ratio_col] = np.where(
            demographic_profile["성연령_문화누리대상자수"].gt(0),
            demographic_profile[count_col] / demographic_profile["성연령_문화누리대상자수"],
            np.nan
        )

    return demographic_profile


if USE_DEMOGRAPHIC_PROFILE:
    demographic_profile = build_demographic_profile(DEMOGRAPHIC_PATH, target_grid)
    profile = profile.merge(demographic_profile, on="취약권역_ID", how="left")

    demographic_matched = profile["성연령_문화누리대상자수"].notna().sum()

    print("성연령 프로파일 결합 권역 수:", demographic_matched)
    print("성연령 프로파일 결합률:", round(demographic_matched / len(profile) * 100, 2), "%")
else:
    print("성별·연령대 프로파일링 제외: USE_DEMOGRAPHIC_PROFILE=False")



## 10. 권역별 프로파일링 보조 변수

- 가장 취약한 시설분류는 중분류별 접근성 취약도가 가장 높은 분류로 판단함.
- 종합취약도 기여 하위지표는 DBSCAN 권역 요약의 `주요취약원인_1`을 사용함.
- 비율 변수는 0으로 나누는 경우 결측으로 유지함.



In [ ]:
profile["프로파일_최취약시설분류"] = (
    profile[facility_category_features]
    .idxmax(axis=1)
    .str.replace("평균_", "", regex=False)
    .str.replace("_접근성취약도", "", regex=False)
)
profile["프로파일_주요기여하위지표"] = profile["주요취약원인_1"]

profile["권역면적_ha"] = profile["권역면적_m2"] / 10000

profile_check_cols = [
    "취약권역_ID",
    "K3_유형",
    "K4_유형",
    "평균_최종취약지수",
    "최고_기준취약점수",
    "포함_취약격자수",
    "권역면적_ha",
    "문화누리대상자_추정인구수",
    "장애인_수요인구수",
    "노령인구_수요인구수",
    "프로파일_최취약시설분류",
    "프로파일_주요기여하위지표"
]

print("권역별 프로파일 예시")
display(profile[profile_check_cols].head(10))



## 11. 유형별 프로파일링

- 유형별 권역 수, 취약점수, 권역 규모, 대상자 규모, 하위지표 취약도를 집계함.
- 유형별 두드러지는 변수는 전체 평균 대비 표준화 차이로 확인함.
- 유형명 확정은 여기서 하지 않고, `K=3`과 `K=4` 프로파일을 비교한 뒤 판단함.



In [ ]:
PROFILE_RATIO_COLS = []


def top_values(series, n=3):
    values = series.dropna().astype(str)
    if values.empty:
        return ""
    return ", ".join(values.value_counts().head(n).index.tolist())


def build_cluster_profile(data, label_col):
    group = data.groupby(label_col, dropna=False)

    summary = group.agg(
        권역수=("취약권역_ID", "nunique"),
        평균_최종취약지수=("평균_최종취약지수", "mean"),
        최고_최종취약지수=("최고_기준취약점수", "max"),
        문화다양성부족도=("평균_문화다양성부족도", "mean"),
        장애인친화취약도=("평균_장애인친화시설_접근성취약도", "mean"),
        노인편의취약도=("평균_노인편의서비스_접근성취약도", "mean"),
        포함_취약격자수=("포함_취약격자수", "sum"),
        평균_권역격자수=("포함_취약격자수", "mean"),
        권역면적_ha=("권역면적_ha", "sum"),
        문화누리대상자수=("문화누리대상자_추정인구수", "sum"),
        장애인수=("장애인_수요인구수", "sum"),
        노령인구수=("노령인구_수요인구수", "sum"),
        문화누리대상자밀도_ha=("문화누리대상자_밀도_ha", "mean")
    ).reset_index()

    summary["장애인비율"] = np.where(
        summary["문화누리대상자수"].gt(0),
        summary["장애인수"] / summary["문화누리대상자수"],
        np.nan
    )
    summary["노령인구비율"] = np.where(
        summary["문화누리대상자수"].gt(0),
        summary["노령인구수"] / summary["문화누리대상자수"],
        np.nan
    )
    summary["주요_시군구"] = summary[label_col].map(group["주요_시군구"].apply(top_values))
    summary["주요_행정동"] = summary[label_col].map(group["주요_행정동"].apply(top_values))
    summary["주요기여하위지표"] = summary[label_col].map(group["프로파일_주요기여하위지표"].apply(top_values))
    summary["최취약시설분류"] = summary[label_col].map(group["프로파일_최취약시설분류"].apply(top_values))

    return summary


def clean_metric_name(name):
    return (
        name
        .replace("평균_", "")
        .replace("_접근성취약도", "접근성취약")
        .replace("_문화누리대상자수", "대상자수")
    )


def build_relative_profile(data, label_col, metric_cols):
    available_cols = [col for col in metric_cols if col in data.columns]
    numeric = data[available_cols].apply(pd.to_numeric, errors="coerce")
    numeric = numeric.fillna(numeric.median(numeric_only=True))

    std = numeric.std(ddof=0).replace(0, np.nan)
    z = (numeric - numeric.mean()) / std
    z = z.fillna(0)
    z[label_col] = data[label_col].to_numpy()

    relative = z.groupby(label_col).mean()
    rows = []

    for cluster_name, row in relative.iterrows():
        high = row.sort_values(ascending=False).head(5)
        low = row.sort_values(ascending=True).head(3)
        rows.append({
            label_col: cluster_name,
            "전체평균보다_높은특성": ", ".join(
                f"{clean_metric_name(col)}({value:+.2f})"
                for col, value in high.items()
            ),
            "전체평균보다_낮은특성": ", ".join(
                f"{clean_metric_name(col)}({value:+.2f})"
                for col, value in low.items()
            )
        })

    return pd.DataFrame(rows)


profile_metric_cols = (
    facility_category_features
    + [
        "평균_문화다양성부족도",
        "평균_장애인친화시설_접근성취약도",
        "평균_노인편의서비스_접근성취약도",
        "평균_최종취약지수",
        "포함_취약격자수",
        "권역면적_ha",
        "문화누리대상자_밀도_ha",
        "장애인비율",
        "노령인구비율"
    ]
    + PROFILE_RATIO_COLS
)

for profile_k in PROFILE_K_LIST:
    label_col = f"K{profile_k}_유형"
    summary = build_cluster_profile(profile, label_col)
    relative = build_relative_profile(profile, label_col, profile_metric_cols)

    print()
    print(f"## K={profile_k} 유형별 프로파일")
    display(summary[[
        label_col,
        "권역수",
        "평균_최종취약지수",
        "최고_최종취약지수",
        "문화다양성부족도",
        "장애인친화취약도",
        "노인편의취약도",
        "포함_취약격자수",
        "권역면적_ha",
        "문화누리대상자수",
        "장애인비율",
        "노령인구비율",
        "주요_시군구",
        "주요기여하위지표",
        "최취약시설분류"
    ]].round({
        "평균_최종취약지수": 2,
        "최고_최종취약지수": 2,
        "문화다양성부족도": 2,
        "장애인친화취약도": 2,
        "노인편의취약도": 2,
        "평균_권역격자수": 1,
        "권역면적_ha": 1,
        "문화누리대상자수": 0,
        "장애인비율": 3,
        "노령인구비율": 3
    }))

    print()
    print(f"## K={profile_k} 전체 평균 대비 특성")
    display(relative)



## 12. 상세 프로파일링 표

- 유형별 분류별 시설 접근성 취약도를 별도 표로 확인함.
- 문화다양성·장애인친화·노인편의 취약도는 하위지표 표로 따로 확인함.
- 이 표들은 해석 확인용이며 별도 파일로 저장하지 않음.



In [ ]:
facility_display_cols = [
    "공연",
    "관광지",
    "도서",
    "문화체험",
    "미술",
    "스포츠관람",
    "영상",
    "음악",
    "체육시설",
    "체육용품"
]

facility_col_map = {
    f"평균_{category}_접근성취약도": category
    for category in facility_display_cols
}

subindicator_col_map = {
    "평균_문화다양성부족도": "문화다양성부족도",
    "평균_장애인친화시설_접근성취약도": "장애인친화취약도",
    "평균_노인편의서비스_접근성취약도": "노인편의취약도",
    "장애인비율": "장애인비율",
    "노령인구비율": "노령인구비율",
}

for profile_k in PROFILE_K_LIST:
    label_col = f"K{profile_k}_유형"

    facility_table = (
        profile
        .groupby(label_col)[list(facility_col_map.keys())]
        .mean()
        .rename(columns=facility_col_map)
        .reset_index()
        .round(3)
    )

    subindicator_table = (
        profile
        .groupby(label_col)[list(subindicator_col_map.keys())]
        .mean()
        .rename(columns=subindicator_col_map)
        .reset_index()
        .round(3)
    )

    print()
    print(f"## K={profile_k} 유형별 분류별 접근성 취약도")
    display(facility_table)

    print()
    print(f"## K={profile_k} 유형별 하위지표 취약도")
    display(subindicator_table)



## 13. K=4 보조 프로파일 저장 옵션

- `K=4`는 과거 비교용 보조 프로파일로만 유지함.
- K-means는 최종 대시보드 유형 분류로 채택하지 않음.
- 기본 설정에서는 권역별 K4 유형 프로파일과 유형별 요약 프로파일을 새로 저장하지 않음.
- 기존 저장 산출물은 K-means 해석 한계 설명용으로만 보존함.

In [ ]:
K4_TYPE_NAME = {
    "K4_유형1": "고취약·문화다양성·생활문화 접근성 부족형",
    "K4_유형2": "고수요·고령장애인 복합수요형",
    "K4_유형3": "체육계열 접근성 취약형",
    "K4_유형4": "미술·공연 특이취약형",
}

K4_TYPE_COLOR = {
    "K4_유형1": "#e4572e",
    "K4_유형2": "#2a9d8f",
    "K4_유형3": "#457b9d",
    "K4_유형4": "#7b2cbf",
}

profile["K4_유형명"] = profile["K4_유형"].map(K4_TYPE_NAME)
profile["K4_유형색상"] = profile["K4_유형"].map(K4_TYPE_COLOR)

k4_region_cols = [
    "취약권역_ID",
    "K4_유형",
    "K4_유형명",
    "K4_유형색상",
    "주요_시군구",
    "주요_행정동",
    "권역중심_x",
    "권역중심_y",
    "포함_취약격자수",
    "권역면적_m2",
    "권역면적_ha",
    "문화누리대상자_추정인구수",
    "장애인_수요인구수",
    "노령인구_수요인구수",
    "장애인비율",
    "노령인구비율",
    "평균_최종취약지수",
    "최고_기준취약점수",
    "평균_문화다양성부족도",
    "평균_장애인친화시설_접근성취약도",
    "평균_노인편의서비스_접근성취약도",
    "프로파일_최취약시설분류",
    "프로파일_주요기여하위지표",
] + facility_category_features

k4_region_profile = profile[k4_region_cols].copy()

k4_type_profile = build_cluster_profile(profile, "K4_유형")
k4_type_profile["K4_유형명"] = k4_type_profile["K4_유형"].map(K4_TYPE_NAME)
k4_type_profile["K4_유형색상"] = k4_type_profile["K4_유형"].map(K4_TYPE_COLOR)

k4_type_facility = (
    profile
    .groupby("K4_유형")[list(facility_col_map.keys())]
    .mean()
    .rename(columns=facility_col_map)
    .add_prefix("시설접근성_")
    .reset_index()
)

k4_type_profile = k4_type_profile.merge(k4_type_facility, on="K4_유형", how="left")

k4_type_order = list(K4_TYPE_NAME.keys())
k4_region_profile["K4_유형"] = pd.Categorical(
    k4_region_profile["K4_유형"],
    categories=k4_type_order,
    ordered=True
)
k4_type_profile["K4_유형"] = pd.Categorical(
    k4_type_profile["K4_유형"],
    categories=k4_type_order,
    ordered=True
)

k4_region_profile = k4_region_profile.sort_values(["K4_유형", "취약권역_ID"]).copy()
k4_type_profile = k4_type_profile.sort_values("K4_유형").copy()

if SAVE_FINAL_TABLES:
    k4_region_path = OUTPUT_PATH / "Kmeans_K4_권역별_유형프로파일.csv"
    k4_type_path = OUTPUT_PATH / "Kmeans_K4_유형별_프로파일.csv"

    k4_region_profile.to_csv(k4_region_path, index=False, encoding="utf-8-sig")
    k4_type_profile.to_csv(k4_type_path, index=False, encoding="utf-8-sig")

    print("저장:", k4_region_path)
    print("저장:", k4_type_path)
else:
    print("최종 테이블 저장 안 함: SAVE_FINAL_TABLES=False")



## 14. 지도 시각화 제외

- DBSCAN 취약권역 지도는 `01_vulnerability_index_build.ipynb`에서 생성함.
- K-means는 최종 대시보드 유형 분류로 채택하지 않으므로 이 노트북에서 지도 이미지를 새로 저장하지 않음.
- 기존 K-means 산출물은 보조 분석 및 한계 설명용으로만 보존함.


In [ ]:
print("지도 이미지 저장 제외")
print("- DBSCAN 상위 10% 취약권역 지도는 01_vulnerability_index_build.ipynb에서 생성함.")
print("- K-means 지도는 최종 대시보드 산출물로 사용하지 않아 새로 저장하지 않음.")


## 15. 프로파일링 결과 점검

- `K=3`은 유형 수가 적어 발표용 구조가 단순함.
- `K=4`는 유형 수가 늘어나지만 이번 분류별 시설 접근성 모델에서는 최소 군집 권역 수가 6개라 단일 이상치 군집은 없음.
- 실루엣 점수만으로는 선명한 군집이라고 보기 어렵기 때문에 최종 K는 유형 설명력과 지도 분포를 함께 보고 결정해야 함.



In [ ]:
print()
print("프로파일링 판단 메모")
print("- K=3과 K=4 모두 최소 군집 권역 수는 6개 이상으로 단일 권역 군집은 없음.")
print("- 분류별 시설 접근성을 넣은 모델은 통합 시설접근성 모델보다 실루엣이 낮아졌으므로, 세부 원인 설명력과 군집 분리력 사이의 절충이 필요함.")
print("- 성별·연령대 변수는 유형 구분력이 약해 이번 프로파일링 출력에서 제외함.")
print("- K=4는 보조 해석용으로만 확인하고, 최종 대시보드 유형으로 확정하지 않음.")
